# Fine-tuning ViT (Google Pre-trained) on Higher Resolution Images

## Task: Binary Classification (Pizza vs Sushi) from Food-101

**Paper Insight:** "When fine-tuning, we tend to use higher resolution images than during pre-training. This is beneficial... However, keeping the patch size the same results in a longer effective sequence length. Vision Transformers can handle arbitrary sequence lengths (up to memory constraints), but **positional embeddings need to be interpolated**." (Dosovitskiy et al., 2021)

This notebook demonstrates:
1.  **Loading Food-101** (Large Images).
2.  **Filtering for 2 Classes** (Pizza & Sushi).
3.  **Using Higher Resolution** (384x384 instead of 224x224).
4.  **Interpolating Positional Embeddings** to make the 224x224 pre-trained model work on 384x384 inputs.

In [ ]:
# Install W&B for tracking
!pip install wandb -q
import wandb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import time
import torch.nn.functional as F

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DL/'
    DATA_PATH = './data' # Download locally in colab initially
    print("✅ Connected to Google Drive")
except ImportError:
    print("❌ Not running on Colab, using local path")
    DATA_PATH = "./data"
    SAVE_PATH = "./saved_models/"
    os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Log in to W&B (Will ask for API Key on first run)
wandb.login()

## 1. Hyperparameters (The "Paper Settings")

-   **Image Size:** `384` (Higher than 224 pre-training)
-   **Batch Size:** Reduced to `32` (Higher res = more VRAM usage)
-   **Classes:** 2 (Pizza, Sushi)

In [ ]:
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
IMG_SIZE = 384  # Higher resolution fine-tuning

## 2. Dataset Preparation (Pizza vs Sushi)

In [ ]:
# ImageNet stats
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

print("📥 Downloading Food-101 (This might take a few minutes for 5GB)... ")
full_train_data = datasets.Food101(root=DATA_PATH, split='train', download=True, transform=train_transform)
full_test_data = datasets.Food101(root=DATA_PATH, split='test', download=True, transform=test_transform)

# --- Filter for Pizza and Sushi ---
def filter_classes(dataset, target_classes):
    indices = []
    for i, label_idx in enumerate(dataset._labels):
         # Access class name strings
        label_name = dataset.classes[label_idx]
        if label_name in target_classes:
            indices.append(i)
    return indices

target_classes = ['pizza', 'sushi']
train_indices = filter_classes(full_train_data, target_classes)
test_indices = filter_classes(full_test_data, target_classes)

train_subset = Subset(full_train_data, train_indices)
test_dataset = Subset(full_test_data, test_indices)

# Create Loaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Filtered Dataset: {len(train_subset)} Train images, {len(test_dataset)} Test images")
print(f"Classes: {target_classes}")

## 3. Load & Interpolate Positional Embeddings

This is the critical step from the paper. Since we changed resolution from 224 to 384, the number of patches changed:
-   **Old:** (224/16)^2 = 14x14 = 196 patches
-   **New:** (384/16)^2 = 24x24 = 576 patches

We must interpolate the 14x14 positional grid to 24x24.

In [ ]:
def load_state_dict_with_resize(model, state_dict):
    """
    Resizes the positional embeddings in state_dict to match the model's new resolution.
    Based on: https://github.com/google-research/vision_transformer/blob/main/vit_jax/checkpoint.py#L198
    """
    # 1. Get the current model's pos_embedding shape
    # Shape: (1, num_patches + 1, embed_dim) -> (1, 577, 768)
    new_pos_embed = model.encoder.pos_embedding.data 
    num_patches_new = new_pos_embed.shape[1] - 1  # 576
    
    # 2. Get the loaded checkpoint's pos_embedding
    # Key is usually 'encoder.pos_embedding' or similar in torchvision models
    old_pos_embed = state_dict['encoder.pos_embedding'] # Shape: (1, 197, 768)
    
    # If shapes match, return immediately (no resizing needed)
    if old_pos_embed.shape == new_pos_embed.shape:
        print("Shapes match, no interpolation needed.")
        return state_dict
        
    print(f"Interpolating Positional Embeddings: {old_pos_embed.shape} -> {new_pos_embed.shape}")

    # 3. Separate CLS token and Grid tokens
    # CLS token is usually at index 0
    old_cls_token = old_pos_embed[:, 0:1, :]
    old_grid_tokens = old_pos_embed[:, 1:, :]
    
    # 4. Reshape grid tokens to 2D image-like grid (B, C, H, W) for interpolation
    # Old grid size: sqrt(196) = 14
    grid_size_old = int(math.sqrt(old_grid_tokens.shape[1])) 
    embed_dim = old_grid_tokens.shape[-1]
    
    # Permute to (B, C, H, W) -> (1, 768, 14, 14)
    old_grid_tokens = old_grid_tokens.permute(0, 2, 1).reshape(1, embed_dim, grid_size_old, grid_size_old)
    
    # 5. Calculate new grid size
    # New grid size: sqrt(576) = 24
    grid_size_new = int(math.sqrt(num_patches_new))
    
    # 6. Interpolate (Bicubic is standard)
    new_grid_tokens = F.interpolate(
        old_grid_tokens, 
        size=(grid_size_new, grid_size_new), 
        mode='bicubic', 
        align_corners=False
    )
    
    # 7. Reshape back to sequence (1, 576, 768) and permute back
    new_grid_tokens = new_grid_tokens.flatten(2).transpose(1, 2)
    
    # 8. Concatenate CLS token back
    new_pos_embed = torch.cat((old_cls_token, new_grid_tokens), dim=1)
    
    # 9. Update state_dict
    state_dict['encoder.pos_embedding'] = new_pos_embed
    
    return state_dict

In [ ]:
# Load standard ViT-Base-16 but DO NOT load weights yet (image_size=384 changes architecture)
model = models.vit_b_16(weights=None, image_size=IMG_SIZE)

# Load standard ImageNet weights manually so we can intercept and resize
pretrained_weights = models.ViT_B_16_Weights.IMAGENET1K_V1.get_state_dict(progress=True)

# INTERPOLATE POSITIONAL EMBEDDINGS
pretrained_weights = load_state_dict_with_resize(model, pretrained_weights)

# Load the modified weights into our 384x384 model
model.load_state_dict(pretrained_weights)

# Modify Head for 2 Classes (Pizza vs Sushi)
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, 2)

model = model.to(device)

## 4. Training Loop

In [ ]:
# Init WandB
wandb.init(
    project="vit-food101-finetune",
    config={
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "architecture": "ViT-B-16"
    }
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

def train(model, loader, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # Map labels: Pizza(index A) -> 0, Sushi(index B) -> 1
        # Since Subset keeps original indices, we need to remap if they are not 0 and 1
        # Food101 indices: Pizza=76, Sushi=96. We need to map 76->0, 96->1.
        # Simplest way: y = (y == sushi_idx).long() 
        pizza_idx = full_train_data.class_to_idx['pizza']
        y = (y != pizza_idx).long() # 0 if pizza, 1 if sushi

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def validate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    pizza_idx = full_train_data.class_to_idx['pizza']
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y = (y != pizza_idx).long() 
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

print("🚀 Starting Higher-Res Fine-tuning (Pizza vs Sushi)...")
for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer)
    val_loss, val_acc = validate(model, test_loader)
    scheduler.step()
    
    # Log to metrics
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "lr": optimizer.param_groups[0]['lr']
    })
    
    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Test Acc: {val_acc:.4f}")

wandb.finish()